<a href="https://colab.research.google.com/github/ZivBNS/Introduction-to-Cloud-Computing/blob/main/targil2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import json
import os

drive.mount('/content/drive')

path = '/content/drive/My Drive/students.json'

Mounted at /content/drive


In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

#  Define the path here if you haven't run Step 1 ---
file_path = '/content/drive/My Drive/students.json'

# Helper to load the latest data from Drive
def get_data():
    # Adding a check to ensure the file exists before reading
    if not os.path.exists(file_path):
        print(f"Error: The file at {file_path} was not found. Please run the initialization block first.")
        return []
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

current_data = get_data()

# Only proceed if data was loaded successfully
if current_data:
    student_options = [f"{s['first_name']} {s['last_name']}" for s in current_data]

    # Define UI Components
    dropdown = widgets.Dropdown(options=student_options, description='Select Student:')
    first_name = widgets.Text(description='First Name:', disabled=True)
    last_name = widgets.Text(description='Last Name:', disabled=True)
    email = widgets.Text(description='Email:', disabled=True)
    courses = widgets.Textarea(description='Courses:', disabled=True)
    link = widgets.Text(description='Link:', disabled=True)
    fav_show = widgets.Text(description='Favorite Show:', placeholder='Enter show name...')
    update_btn = widgets.Button(description='Update Show', button_style='info')
    output_log = widgets.Output()

    # Logic to update fields when selection changes
    def update_fields(change):
        if change['type'] == 'change' and change['name'] == 'value':
            student = next(s for s in current_data if f"{s['first_name']} {s['last_name']}" == change['new'])
            first_name.value = student.get('first_name', '')
            last_name.value = student.get('last_name', '')
            email.value = student.get('email', '')
            courses.value = ", ".join(student.get('courses', []))
            link.value = student.get('link', '')
            fav_show.value = student.get('favorite_show', '')

    dropdown.observe(update_fields)

    # Logic for the Update Button
    def on_button_clicked(b):
        with output_log:
            clear_output()
            selected_name = dropdown.value
            for student in current_data:
                if f"{student['first_name']} {student['last_name']}" == selected_name:
                    student['favorite_show'] = fav_show.value
                    break

            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(current_data, f, indent=4)
            print(f"Success! Updated {selected_name}'s favorite show.")

    update_btn.on_click(on_button_clicked)

    # Layout and Display
    form_layout = widgets.VBox([
        widgets.HTML("<h2>Student Information Management</h2>"),
        dropdown,
        widgets.HBox([first_name, last_name]),
        email,
        courses,
        link,
        widgets.HTML("<hr>"),
        fav_show,
        update_btn,
        output_log
    ])

    # Initialize the fields for the first student
    update_fields({'type': 'change', 'name': 'value', 'new': dropdown.value})
    display(form_layout)